# 🌿 Leaf Disease Detection — Complete Notebook
### Training + Prediction + Pesticide Recommendations

---

## 📋 Step-by-Step Guide (Read Before Running)

### Step 1 — Set Runtime to GPU
- Go to **Runtime → Change runtime type → T4 GPU** → Save
- Without GPU, training will take 10x longer

### Step 2 — Upload your Kaggle API key
- Go to https://www.kaggle.com/settings → API → **Create New Token**
- A file called `kaggle.json` will download
- Upload it when Cell 2 asks you

### Step 3 — Run cells one by one (top to bottom)
- **Cells 1–5**: Setup and dataset download (~5–10 min)
- **Cells 6–10**: Build and train model (~35–50 min on T4 GPU)
- **Cells 11–12**: Evaluate and save model
- **Cell 13**: Download your trained model
- **Cell 14**: Test prediction with disease + pesticide recommendations

### Step 4 — After training
- Download `plant_disease_model_final.h5` and `class_names.json`
- Upload both to your HuggingFace Space
- Update `app.py` preprocessing (shown at the end)

---
**Expected training time:** 35–50 min on T4 GPU  
**Expected accuracy:** 93–96%  
**Dataset:** PlantVillage (87,000+ images, 38 classes)

In [ ]:
# ============================================================
# CELL 1 — Check GPU
# ============================================================
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'✅ GPU Found: {gpus}')
    print('Good to go!')
else:
    print('⚠️ No GPU found!')
    print('Go to Runtime → Change runtime type → T4 GPU')

print('TensorFlow version:', tf.__version__)

In [ ]:
# ============================================================
# CELL 2 — Download PlantVillage Dataset from Kaggle
# ============================================================
# You need a free Kaggle account.
# Go to https://www.kaggle.com/settings → API → Create New Token
# Upload the kaggle.json file that downloads

!pip install kaggle -q

from google.colab import files
print('📁 Upload your kaggle.json file now:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print('\n⬇️ Downloading PlantVillage dataset...')
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -q
print('✅ Download complete!')

print('\n📦 Extracting...')
!unzip -q new-plant-diseases-dataset.zip -d plantvillage
print('✅ Extracted!')

In [ ]:
# ============================================================
# CELL 3 — Find dataset folders automatically
# ============================================================
import os

BASE = 'plantvillage'

candidates = [
    'plantvillage/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train',
    'plantvillage/train',
    'plantvillage/dataset/train',
]

TRAIN_DIR = None
VALID_DIR = None

for c in candidates:
    if os.path.exists(c):
        TRAIN_DIR = c
        VALID_DIR = c.replace('train', 'valid')
        if not os.path.exists(VALID_DIR):
            VALID_DIR = c.replace('train', 'val')
        break

if TRAIN_DIR is None:
    print('Could not auto-detect. Listing folders:')
    for r, d, f in os.walk(BASE):
        if len(d) > 5:
            print(r, '->', sorted(d)[:5], '...')
            break
    raise ValueError('Set TRAIN_DIR and VALID_DIR manually')

print('✅ Train dir:', TRAIN_DIR)
print('✅ Valid dir:', VALID_DIR)
print('📊 Classes found:', len(os.listdir(TRAIN_DIR)))

# Count total images
total = sum(len(files) for _, _, files in os.walk(TRAIN_DIR))
print(f'🖼️  Total training images: {total:,}')

In [ ]:
# ============================================================
# CELL 4 — Configuration
# ============================================================

IMG_SIZE         = 224
BATCH_SIZE       = 32     # reduce to 16 if you get memory errors
EPOCHS           = 15     # EarlyStopping will stop earlier if needed
FINE_TUNE_EPOCHS = 10     # Phase 2 fine-tuning
LEARNING_RATE    = 1e-3
FINE_TUNE_LR     = 1e-5
MODEL_SAVE_PATH  = 'plant_disease_model_final.h5'

print('Config:')
print(f'  Image size  : {IMG_SIZE}x{IMG_SIZE}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Phase 1     : max {EPOCHS} epochs (EarlyStopping active)')
print(f'  Phase 2     : max {FINE_TUNE_EPOCHS} fine-tune epochs')
print(f'  Model saved : {MODEL_SAVE_PATH}')

In [ ]:
# ============================================================
# CELL 5 — Data generators with augmentation
# ============================================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training: augmentation makes model robust to real-world photos
# (different angles, lighting, phone cameras)
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],   # simulate different lighting conditions
    fill_mode='reflect'
)

# Validation: no augmentation, only preprocessing
valid_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

valid_gen = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

NUM_CLASSES = len(train_gen.class_indices)
CLASS_NAMES = list(train_gen.class_indices.keys())

print(f'✅ Total classes: {NUM_CLASSES}')
print(f'✅ Train batches: {len(train_gen)}')
print(f'✅ Valid batches: {len(valid_gen)}')
print('Sample classes:', CLASS_NAMES[:5], '...')

In [ ]:
# ============================================================
# CELL 6 — Class weights (handles unbalanced data)
# ============================================================
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

labels = train_gen.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weight_dict = dict(enumerate(class_weights))

print('✅ Class weights computed')
print(f'   Min weight: {min(class_weights):.3f}')
print(f'   Max weight: {max(class_weights):.3f}')
print('   (Unbalanced classes will be handled fairly during training)')

In [ ]:
# ============================================================
# CELL 7 — Build model (EfficientNetB3)
# ============================================================
# WHY EfficientNetB3 instead of MobileNetV2?
# MobileNetV2 accuracy on PlantVillage: ~78-83%
# EfficientNetB3 accuracy on PlantVillage: ~93-96%
# EfficientNetB3 is only slightly slower but much more accurate

from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras import layers, Model

# Load pretrained base (ImageNet weights)
base_model = EfficientNetB3(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False   # freeze base for Phase 1

# Custom classification head
inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(512, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

total_params     = model.count_params()
trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'✅ Model built')
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {trainable_params:,} (head only for now)')

In [ ]:
# ============================================================
# CELL 8 — Phase 1: Train classification head
# ============================================================
# In Phase 1 the base model (EfficientNetB3) is FROZEN.
# Only our Dense layers train. This is fast and stable.

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import time

callbacks_p1 = [
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=3, min_lr=1e-7, verbose=1
    ),
    ModelCheckpoint(
        MODEL_SAVE_PATH, monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print('=' * 50)
print('PHASE 1: Training classification head only')
print(f'Max {EPOCHS} epochs — EarlyStopping will stop earlier if needed')
print('Base model: FROZEN')
print('=' * 50)

t0 = time.time()
history1 = model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=EPOCHS,
    callbacks=callbacks_p1,
    class_weight=class_weight_dict,
    verbose=1
)
t1 = time.time()

p1_epochs = len(history1.history['accuracy'])
p1_best   = max(history1.history['val_accuracy'])
print(f'\nPhase 1 done in {(t1-t0)/60:.1f} min')
print(f'Epochs run     : {p1_epochs}')
print(f'Best val acc   : {p1_best*100:.2f}%')

In [ ]:
# ============================================================
# CELL 9 — Phase 2: Fine-tune top layers
# ============================================================
# Unfreeze last 30 layers of EfficientNetB3.
# Use very low LR so we don't destroy ImageNet features.

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen layers in base model: {unfrozen}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=3, min_lr=1e-8, verbose=1
    ),
    ModelCheckpoint(
        MODEL_SAVE_PATH, monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print('=' * 50)
print('PHASE 2: Fine-tuning (last 30 EfficientNet layers)')
print(f'Max {FINE_TUNE_EPOCHS} epochs — LR: {FINE_TUNE_LR}')
print('=' * 50)

t2 = time.time()
history2 = model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks_p2,
    class_weight=class_weight_dict,
    verbose=1
)
t3 = time.time()

p2_epochs = len(history2.history['accuracy'])
p2_best   = max(history2.history['val_accuracy'])
print(f'\nPhase 2 done in {(t3-t2)/60:.1f} min')
print(f'Epochs run    : {p2_epochs}')
print(f'Best val acc  : {p2_best*100:.2f}%')
print(f'Total training: {(t3-t0)/60:.1f} min')

In [ ]:
# ============================================================
# CELL 10 — Plot training curves
# ============================================================
import matplotlib.pyplot as plt

# Combine both phases
all_acc  = history1.history['accuracy']  + history2.history['accuracy']
all_vacc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss']  + history2.history['loss']
all_vloss= history1.history['val_loss'] + history2.history['val_loss']
ep_range = range(1, len(all_acc) + 1)
split_ep = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep_range, all_acc,  'b-o', label='Train', markersize=4)
axes[0].plot(ep_range, all_vacc, 'g-o', label='Validation', markersize=4)
axes[0].axvline(split_ep + 0.5, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
axes[0].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

axes[1].plot(ep_range, all_loss,  'b-o', label='Train', markersize=4)
axes[1].plot(ep_range, all_vloss, 'r-o', label='Validation', markersize=4)
axes[1].axvline(split_ep + 0.5, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
axes[1].set_title('Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
# ============================================================
# CELL 11 — Final evaluation on validation set
# ============================================================
print('Evaluating on validation set...')
loss, acc = model.evaluate(valid_gen, verbose=1)
print(f'\n✅ Final Results:')
print(f'   Validation Loss    : {loss:.4f}')
print(f'   Validation Accuracy: {acc*100:.2f}%')

if acc >= 0.93:
    print('\n🎉 Excellent! Model is ready for deployment.')
elif acc >= 0.85:
    print('\n✅ Good accuracy. Model is usable.')
else:
    print('\n⚠️ Lower than expected. Try increasing epochs or check dataset.')

In [ ]:
# ============================================================
# CELL 12 — Save class names
# ============================================================
import json

with open('class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

print('✅ Saved class_names.json')
print(f'   Total classes: {len(CLASS_NAMES)}')
print('\nAll class names:')
for i, name in enumerate(CLASS_NAMES):
    tag = '🌿' if 'healthy' in name.lower() else '🦠'
    print(f'  {i:2d}: {tag} {name}')

In [ ]:
# ============================================================
# CELL 13 — Download model files
# ============================================================
from google.colab import files
import os

model_mb = os.path.getsize(MODEL_SAVE_PATH) / (1024 * 1024)
print(f'Model file: {MODEL_SAVE_PATH} ({model_mb:.1f} MB)')

print('\n⬇️ Downloading plant_disease_model_final.h5 ...')
files.download(MODEL_SAVE_PATH)

print('⬇️ Downloading class_names.json ...')
files.download('class_names.json')

# Also backup to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!cp {MODEL_SAVE_PATH} "/content/drive/MyDrive/plant_disease_model_final.h5"
!cp class_names.json "/content/drive/MyDrive/class_names.json"
print('\n✅ Also backed up to Google Drive!')

print('\n' + '='*50)
print('NEXT STEPS:')
print('1. Upload plant_disease_model_final.h5 to HuggingFace Space')
print('2. Upload class_names.json to HuggingFace Space')
print('3. Update preprocessing in app.py (see Cell 14 notes)')
print('='*50)

---
## 🔍 Prediction + Pesticide Recommendations
### Run Cell 14 after training is done (or load an existing model)

In [ ]:
# ============================================================
# CELL 14 — Full Disease Info: Cause + Pesticide Recommendations
# ============================================================
# Complete pesticide recommendations for all 38 PlantVillage classes.

DISEASE_INFO = {

    # ── APPLE ─────────────────────────────────────────────────
    "Apple___Apple_scab": {
        "simple_name":  "Apple Scab",
        "pathogen_type": "Fungus",
        "pathogen":     "Venturia inaequalis",
        "cause":        "Fungal spores spread in wet, cool spring weather",
        "region":       "North India (Himachal Pradesh, J&K)",
        "season":       "Spring to early summer",
        "symptoms":     "Olive-green to black spots on leaves and fruit",
        "pesticide": [
            {"product": "Mancozeb 75% WP",      "dose": "2.5 g/L water",  "when": "Spray at first sign of spots"},
            {"product": "Carbendazim 50% WP",   "dose": "1 g/L water",    "when": "Repeat every 10–12 days"},
            {"product": "Captan 50% WP",         "dose": "2 g/L water",    "when": "Preventive spray before rain"},
        ],
        "organic_option": "Neem oil 5ml/L + copper sulphate spray",
        "base_days": 8
    },
    "Apple___Black_rot": {
        "simple_name":  "Apple Black Rot",
        "pathogen_type": "Fungus",
        "pathogen":     "Botryosphaeria obtusa",
        "cause":        "Fungal infection spreading through wounds and pruning cuts",
        "region":       "North India",
        "season":       "Summer–Monsoon",
        "symptoms":     "Brown-purple leaf spots with yellow border; black fruit rot",
        "pesticide": [
            {"product": "Captan 50% WP",         "dose": "2 g/L water",    "when": "Every 10 days during wet season"},
            {"product": "Mancozeb 75% WP",       "dose": "2.5 g/L water",  "when": "At first sign of disease"},
            {"product": "Thiophanate-methyl",    "dose": "1 g/L water",    "when": "For systemic control"},
        ],
        "organic_option": "Bordeaux mixture (copper sulphate + lime)",
        "base_days": 12
    },
    "Apple___Cedar_apple_rust": {
        "simple_name":  "Cedar Apple Rust",
        "pathogen_type": "Fungus",
        "pathogen":     "Gymnosporangium juniperi-virginianae",
        "cause":        "Fungus alternates between apple and cedar/juniper trees",
        "region":       "Himalayan region",
        "season":       "Spring",
        "symptoms":     "Bright orange-yellow spots on upper leaf surface",
        "pesticide": [
            {"product": "Myclobutanil 10% WP",   "dose": "1 g/L water",    "when": "At bud break and every 10 days"},
            {"product": "Triadimefon 25% WP",    "dose": "0.5 g/L water",  "when": "Preventive during spring"},
            {"product": "Propiconazole 25% EC",  "dose": "1 ml/L water",   "when": "At first symptom"},
        ],
        "organic_option": "Sulphur dust 3 kg/acre during spring",
        "base_days": 10
    },
    "Apple___healthy": {
        "simple_name": "Healthy Apple", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease detected",
        "region": "All", "season": "All year",
        "symptoms": "No symptoms",
        "pesticide": [
            {"product": "Preventive: Bordeaux mixture", "dose": "1% solution", "when": "Once before rainy season"}
        ],
        "organic_option": "Neem oil monthly spray as prevention",
        "base_days": 30
    },

    # ── BLUEBERRY ─────────────────────────────────────────────
    "Blueberry___healthy": {
        "simple_name": "Healthy Blueberry", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease detected",
        "region": "All", "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Keep soil pH 4.5–5.5 for healthy growth",
        "base_days": 30
    },

    # ── CHERRY ────────────────────────────────────────────────
    "Cherry_(including_sour)___Powdery_mildew": {
        "simple_name":  "Cherry Powdery Mildew",
        "pathogen_type": "Fungus",
        "pathogen":     "Podosphaera clandestina",
        "cause":        "Dry warm days with cool humid nights favour this fungus",
        "region":       "North India",
        "season":       "Late spring to summer",
        "symptoms":     "White powdery coating on young leaves and shoots",
        "pesticide": [
            {"product": "Sulphur 80% WP",          "dose": "2 g/L water",   "when": "First sign of white powder"},
            {"product": "Triadimefon 25% WP",       "dose": "0.5 g/L water","when": "Repeat every 14 days"},
            {"product": "Hexaconazole 5% SC",       "dose": "2 ml/L water",  "when": "For severe infection"},
        ],
        "organic_option": "Baking soda solution (1 tsp/L) + neem oil",
        "base_days": 7
    },
    "Cherry_(including_sour)___healthy": {
        "simple_name": "Healthy Cherry", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Regular pruning for air circulation",
        "base_days": 30
    },

    # ── CORN / MAIZE ──────────────────────────────────────────
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": {
        "simple_name":  "Corn Gray Leaf Spot",
        "pathogen_type": "Fungus",
        "pathogen":     "Cercospora zeae-maydis",
        "cause":        "Humid weather and dense planting spread spores",
        "region":       "All India",
        "season":       "Kharif (Monsoon)",
        "symptoms":     "Rectangular grey to tan lesions on lower leaves",
        "pesticide": [
            {"product": "Mancozeb 75% WP",       "dose": "2.5 g/L water",  "when": "At first sign"},
            {"product": "Azoxystrobin 23% SC",   "dose": "1 ml/L water",   "when": "Every 14 days"},
            {"product": "Propiconazole 25% EC",  "dose": "1 ml/L water",   "when": "Severe infection"},
        ],
        "organic_option": "Plant resistant varieties; crop rotation",
        "base_days": 10
    },
    "Corn_(maize)___Common_rust_": {
        "simple_name":  "Corn Common Rust",
        "pathogen_type": "Fungus",
        "pathogen":     "Puccinia sorghi",
        "cause":        "Wind-dispersed spores; favoured by cool humid weather",
        "region":       "All India",
        "season":       "Monsoon",
        "symptoms":     "Small golden-brown pustules on both leaf surfaces",
        "pesticide": [
            {"product": "Mancozeb 75% WP",        "dose": "2.5 g/L water",  "when": "At first pustule"},
            {"product": "Zineb 68% WP",            "dose": "2 g/L water",    "when": "Every 10 days"},
            {"product": "Tebuconazole 25.9% EC",   "dose": "1 ml/L water",   "when": "Severe cases"},
        ],
        "organic_option": "Use rust-resistant hybrid seed varieties",
        "base_days": 10
    },
    "Corn_(maize)___Northern_Leaf_Blight": {
        "simple_name":  "Corn Northern Leaf Blight",
        "pathogen_type": "Fungus",
        "pathogen":     "Exserohilum turcicum",
        "cause":        "Cool, wet weather promotes fungal growth",
        "region":       "North and Central India",
        "season":       "Monsoon",
        "symptoms":     "Long cigar-shaped grey-green lesions on leaves",
        "pesticide": [
            {"product": "Mancozeb 75% WP",        "dose": "2.5 g/L water",  "when": "Early infection stage"},
            {"product": "Propiconazole 25% EC",   "dose": "1 ml/L water",   "when": "Every 14 days"},
            {"product": "Chlorothalonil 75% WP",  "dose": "2 g/L water",    "when": "Preventive spray"},
        ],
        "organic_option": "Deep ploughing to bury infected stubble",
        "base_days": 10
    },
    "Corn_(maize)___healthy": {
        "simple_name": "Healthy Maize", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Crop rotation with legumes",
        "base_days": 30
    },

    # ── GRAPE ─────────────────────────────────────────────────
    "Grape___Black_rot": {
        "simple_name":  "Grape Black Rot",
        "pathogen_type": "Fungus",
        "pathogen":     "Guignardia bidwellii",
        "cause":        "Wet spring weather; spores released from infected canes",
        "region":       "Maharashtra, Karnataka, Tamil Nadu",
        "season":       "Pre-monsoon to Monsoon",
        "symptoms":     "Tan-brown leaf spots with black border; shrivelled black berries",
        "pesticide": [
            {"product": "Mancozeb 75% WP",        "dose": "2.5 g/L water",  "when": "Spray 3 times at 10-day intervals"},
            {"product": "Captan 50% WP",           "dose": "2 g/L water",    "when": "Preventive from bud break"},
            {"product": "Myclobutanil 10% WP",     "dose": "1 g/L water",    "when": "After first symptoms"},
        ],
        "organic_option": "Bordeaux mixture 1% before first rain",
        "base_days": 10
    },
    "Grape___Esca_(Black_Measles)": {
        "simple_name":  "Grape Esca (Black Measles)",
        "pathogen_type": "Fungus",
        "pathogen":     "Phaeomoniella chlamydospora / Phaeoacremonium spp.",
        "cause":        "Fungal wood disease entering through pruning wounds",
        "region":       "All grape-growing regions",
        "season":       "Summer",
        "symptoms":     "Tiger-stripe leaf discoloration; internal wood browning",
        "pesticide": [
            {"product": "Thiophanate-methyl 70% WP", "dose": "1.5 g/L water", "when": "Apply on pruning cuts immediately"},
            {"product": "Fosetyl-Al 80% WP",          "dose": "2.5 g/L water", "when": "Foliar spray at symptom onset"},
        ],
        "organic_option": "Paint pruning wounds with Trichoderma paste",
        "base_days": 14
    },
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": {
        "simple_name":  "Grape Leaf Blight",
        "pathogen_type": "Fungus",
        "pathogen":     "Pseudocercospora vitis (Isariopsis clavispora)",
        "cause":        "Warm humid conditions; windborne spores",
        "region":       "Maharashtra, Karnataka",
        "season":       "Monsoon",
        "symptoms":     "Reddish-brown irregular spots on upper leaf surface",
        "pesticide": [
            {"product": "Mancozeb 75% WP",        "dose": "2.5 g/L water",  "when": "Early symptom stage"},
            {"product": "Carbendazim 50% WP",      "dose": "1 g/L water",    "when": "Every 10–14 days"},
        ],
        "organic_option": "Neem oil + garlic extract spray",
        "base_days": 10
    },
    "Grape___healthy": {
        "simple_name": "Healthy Grape", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Regular canopy management for air circulation",
        "base_days": 30
    },

    # ── ORANGE ────────────────────────────────────────────────
    "Orange___Haunglongbing_(Citrus_greening)": {
        "simple_name":  "Citrus Greening (HLB)",
        "pathogen_type": "Bacteria",
        "pathogen":     "Candidatus Liberibacter asiaticus",
        "cause":        "Spread by Asian citrus psyllid insect (Diaphorina citri)",
        "region":       "Maharashtra, Andhra Pradesh, Tamil Nadu",
        "season":       "All year (psyllid active in warm weather)",
        "symptoms":     "Yellow mottled leaves (blotchy mottle); misshapen bitter fruit",
        "pesticide": [
            {"product": "Imidacloprid 17.8% SL (for psyllid)", "dose": "0.5 ml/L water", "when": "Spray for psyllid control every 21 days"},
            {"product": "Dimethoate 30% EC (for psyllid)",      "dose": "2 ml/L water",   "when": "Alternate with imidacloprid"},
            {"product": "Oxytetracycline (antibiotic, limited)","dose": "As per label",   "when": "Early stage trunk injection only"},
        ],
        "organic_option": "Remove infected trees; release Tamarixia radiata (biological psyllid control agent)",
        "base_days": 3
    },

    # ── PEACH ─────────────────────────────────────────────────
    "Peach___Bacterial_spot": {
        "simple_name":  "Peach Bacterial Spot",
        "pathogen_type": "Bacteria",
        "pathogen":     "Xanthomonas arboricola pv. pruni",
        "cause":        "Rain splash spreads bacteria to leaves and fruit",
        "region":       "North India (Himachal Pradesh)",
        "season":       "Spring to summer",
        "symptoms":     "Small water-soaked spots turning brown; shot-hole effect",
        "pesticide": [
            {"product": "Copper hydroxide 77% WP",   "dose": "2 g/L water",  "when": "Spray before rainy periods"},
            {"product": "Streptomycin sulphate",      "dose": "0.2 g/L water","when": "At first bacterial symptom"},
            {"product": "Bordeaux mixture 1%",        "dose": "1% solution",  "when": "Preventive after pruning"},
        ],
        "organic_option": "Copper-based sprays; avoid overhead irrigation",
        "base_days": 7
    },
    "Peach___healthy": {
        "simple_name": "Healthy Peach", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Proper drainage to prevent root diseases",
        "base_days": 30
    },

    # ── BELL PEPPER ───────────────────────────────────────────
    "Pepper,_bell___Bacterial_spot": {
        "simple_name":  "Bell Pepper Bacterial Spot",
        "pathogen_type": "Bacteria",
        "pathogen":     "Xanthomonas campestris pv. vesicatoria",
        "cause":        "Warm wet weather; infected seed or transplants",
        "region":       "All India",
        "season":       "Monsoon",
        "symptoms":     "Water-soaked spots turning brown with yellow halo",
        "pesticide": [
            {"product": "Copper oxychloride 50% WP",  "dose": "2.5 g/L water", "when": "At first sign"},
            {"product": "Streptomycin 90% + Tetracycline 10%", "dose": "0.5 g/L", "when": "Severe infection"},
            {"product": "Kasugamycin 3% SL",            "dose": "2 ml/L water",  "when": "Every 7 days"},
        ],
        "organic_option": "Hot water seed treatment (50°C, 25 min); neem cake in soil",
        "base_days": 7
    },
    "Pepper,_bell___healthy": {
        "simple_name": "Healthy Bell Pepper", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Mulching to reduce soil splash",
        "base_days": 30
    },

    # ── POTATO ────────────────────────────────────────────────
    "Potato___Early_blight": {
        "simple_name":  "Potato Early Blight",
        "pathogen_type": "Fungus",
        "pathogen":     "Alternaria solani",
        "cause":        "Warm temperatures and leaf wetness",
        "region":       "All India",
        "season":       "Summer–Monsoon",
        "symptoms":     "Dark brown target-like rings on older leaves",
        "pesticide": [
            {"product": "Mancozeb 75% WP",         "dose": "2.5 g/L water",  "when": "First sign; repeat every 7–10 days"},
            {"product": "Chlorothalonil 75% WP",   "dose": "2 g/L water",    "when": "Alternate with mancozeb"},
            {"product": "Azoxystrobin 23% SC",     "dose": "1 ml/L water",   "when": "Systemic control"},
        ],
        "organic_option": "Neem oil 5ml/L; remove and destroy infected lower leaves",
        "base_days": 10
    },
    "Potato___Late_blight": {
        "simple_name":  "Potato Late Blight",
        "pathogen_type": "Oomycete (water mould)",
        "pathogen":     "Phytophthora infestans",
        "cause":        "Cool wet weather (15–18°C); rain spreads spores rapidly",
        "region":       "North India, hilly areas",
        "season":       "Monsoon",
        "symptoms":     "Water-soaked dark lesions with white mould under leaf",
        "pesticide": [
            {"product": "Metalaxyl 8% + Mancozeb 64% WP", "dose": "2.5 g/L water", "when": "URGENT: Spray within 2–3 days"},
            {"product": "Cymoxanil 8% + Mancozeb 64% WP", "dose": "2.5 g/L water", "when": "Alternate every 7 days"},
            {"product": "Dimethomorph 50% WP",             "dose": "1 g/L water",   "when": "Severe cases"},
        ],
        "organic_option": "Bordeaux mixture 1%; destroy infected plants immediately",
        "base_days": 3
    },
    "Potato___healthy": {
        "simple_name": "Healthy Potato", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Certified disease-free seed tubers",
        "base_days": 30
    },

    # ── RASPBERRY, SOYBEAN ────────────────────────────────────
    "Raspberry___healthy": {
        "simple_name": "Healthy Raspberry", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Proper cane management and spacing",
        "base_days": 30
    },
    "Soybean___healthy": {
        "simple_name": "Healthy Soybean", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Rhizobium inoculation for healthy growth",
        "base_days": 30
    },

    # ── SQUASH ────────────────────────────────────────────────
    "Squash___Powdery_mildew": {
        "simple_name":  "Squash Powdery Mildew",
        "pathogen_type": "Fungus",
        "pathogen":     "Podosphaera xanthii / Erysiphe cichoracearum",
        "cause":        "Dry warm conditions with high humidity nights",
        "region":       "All India",
        "season":       "Summer to early Monsoon",
        "symptoms":     "White powdery patches on upper leaf surface",
        "pesticide": [
            {"product": "Sulphur 80% WP",          "dose": "2.5 g/L water", "when": "First sign of white patches"},
            {"product": "Triadimefon 25% WP",       "dose": "0.5 g/L water","when": "Repeat every 10 days"},
            {"product": "Dinocap 48% EC",            "dose": "1 ml/L water",  "when": "Heavy infection"},
        ],
        "organic_option": "Milk spray (40% diluted milk) or baking soda solution",
        "base_days": 7
    },

    # ── STRAWBERRY ────────────────────────────────────────────
    "Strawberry___Leaf_scorch": {
        "simple_name":  "Strawberry Leaf Scorch",
        "pathogen_type": "Fungus",
        "pathogen":     "Diplocarpon earlianum",
        "cause":        "Cool wet weather; spores splash-spread from soil to leaves",
        "region":       "Himachal Pradesh, Uttarakhand, Kashmir",
        "season":       "Spring to early summer (cool & wet)",
        "symptoms":     "Irregular purplish-brown spots on leaf surface; scorched leaf edges and tips turning brown",
        "pesticide": [
            {"product": "Captan 50% WP",           "dose": "2 g/L water",    "when": "At first brown spots; repeat every 7–10 days"},
            {"product": "Myclobutanil 10% WP",     "dose": "1 g/L water",    "when": "Systemic control; every 14 days"},
            {"product": "Azoxystrobin 23% SC",     "dose": "1 ml/L water",   "when": "Alternate with captan"},
        ],
        "organic_option": "Remove infected leaves; neem oil spray 5ml/L weekly",
        "base_days": 7
    },
    "Strawberry___healthy": {
        "simple_name": "Healthy Strawberry", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Mulch with straw to reduce soil splash",
        "base_days": 30
    },

    # ── TOMATO ────────────────────────────────────────────────
    "Tomato___Bacterial_spot": {
        "simple_name":  "Tomato Bacterial Spot",
        "pathogen_type": "Bacteria",
        "pathogen":     "Xanthomonas campestris pv. vesicatoria",
        "cause":        "Warm rainy weather; infected transplants",
        "region":       "All India",
        "season":       "Monsoon",
        "symptoms":     "Small dark water-soaked spots with yellow halo",
        "pesticide": [
            {"product": "Copper oxychloride 50% WP",  "dose": "3 g/L water",  "when": "At first sign; every 7 days"},
            {"product": "Streptomycin sulphate 90% SP","dose": "0.2 g/L water","when": "For severe cases"},
        ],
        "organic_option": "Seed treatment with hot water (50°C, 25 min)",
        "base_days": 7
    },
    "Tomato___Early_blight": {
        "simple_name":  "Tomato Early Blight",
        "pathogen_type": "Fungus",
        "pathogen":     "Alternaria solani",
        "cause":        "Warm humid weather; infected crop debris in soil",
        "region":       "All India",
        "season":       "Summer–Monsoon",
        "symptoms":     "Concentric ring-like brown spots (target board pattern) on older leaves",
        "pesticide": [
            {"product": "Mancozeb 75% WP",          "dose": "2.5 g/L water",  "when": "First sign; repeat every 7–10 days"},
            {"product": "Chlorothalonil 75% WP",    "dose": "2 g/L water",    "when": "Alternate with mancozeb"},
            {"product": "Iprodione 50% WP",          "dose": "1.5 g/L water",  "when": "Severe infection"},
        ],
        "organic_option": "Neem oil + baking soda spray; remove lower infected leaves",
        "base_days": 10
    },
    "Tomato___Late_blight": {
        "simple_name":  "Tomato Late Blight",
        "pathogen_type": "Oomycete (water mould)",
        "pathogen":     "Phytophthora infestans",
        "cause":        "Cool wet weather; extremely fast spreading",
        "region":       "North India, hilly regions",
        "season":       "Monsoon",
        "symptoms":     "Dark water-soaked lesions with white cottony growth on underside",
        "pesticide": [
            {"product": "Metalaxyl 8% + Mancozeb 64% WP", "dose": "2.5 g/L water", "when": "URGENT within 2–3 days"},
            {"product": "Dimethomorph 50% WP",             "dose": "1 g/L water",   "when": "Alternate every 7 days"},
            {"product": "Famoxadone 16.6% + Cymoxanil 22.1%", "dose": "0.6 g/L",   "when": "Severe epidemic"},
        ],
        "organic_option": "Bordeaux mixture 1%; destroy infected plants immediately",
        "base_days": 3
    },
    "Tomato___Leaf_Mold": {
        "simple_name":  "Tomato Leaf Mold",
        "pathogen_type": "Fungus",
        "pathogen":     "Passalora fulva (Cladosporium fulvum)",
        "cause":        "High humidity (>85%); greenhouse crops most affected",
        "region":       "All India",
        "season":       "Monsoon",
        "symptoms":     "Pale green/yellow spots on upper leaf; olive-brown mould below",
        "pesticide": [
            {"product": "Mancozeb 75% WP",          "dose": "2.5 g/L water",  "when": "At first spots"},
            {"product": "Chlorothalonil 75% WP",    "dose": "2 g/L water",    "when": "Every 7 days"},
            {"product": "Hexaconazole 5% SC",        "dose": "2 ml/L water",   "when": "Severe cases"},
        ],
        "organic_option": "Improve ventilation; reduce humidity in greenhouse",
        "base_days": 7
    },
    "Tomato___Septoria_leaf_spot": {
        "simple_name":  "Tomato Septoria Leaf Spot",
        "pathogen_type": "Fungus",
        "pathogen":     "Septoria lycopersici",
        "cause":        "Wet weather; spores from infected soil splash onto leaves",
        "region":       "All India",
        "season":       "Monsoon",
        "symptoms":     "Small circular spots with dark border and grey center, black dots inside",
        "pesticide": [
            {"product": "Mancozeb 75% WP",          "dose": "2.5 g/L water",  "when": "First sign; every 7–10 days"},
            {"product": "Copper oxychloride 50% WP","dose": "3 g/L water",    "when": "Alternate spray"},
            {"product": "Chlorothalonil 75% WP",    "dose": "2 g/L water",    "when": "During heavy rain"},
        ],
        "organic_option": "Mulching; avoid overhead watering; remove infected leaves",
        "base_days": 7
    },
    "Tomato___Spider_mites Two-spotted_spider_mite": {
        "simple_name":  "Tomato Spider Mites",
        "pathogen_type": "Pest (Arachnid)",
        "pathogen":     "Tetranychus urticae (Two-spotted spider mite)",
        "cause":        "Hot dry weather; mites suck cell sap from undersides of leaves",
        "region":       "All India",
        "season":       "Summer (hot & dry)",
        "symptoms":     "Tiny yellow/white stippling on leaves; fine webbing on underside",
        "pesticide": [
            {"product": "Abamectin 1.9% EC (acaricide)",  "dose": "0.5 ml/L water", "when": "First sign of stippling"},
            {"product": "Spiromesifen 22.9% SC",           "dose": "1 ml/L water",   "when": "Repeat after 10 days"},
            {"product": "Dicofol 18.5% EC",               "dose": "2 ml/L water",   "when": "Heavy infestation"},
        ],
        "organic_option": "Neem oil 5ml/L; strong water jet on leaf undersides; release Phytoseiulus persimilis (predatory mite)",
        "base_days": 5
    },
    "Tomato___Target_Spot": {
        "simple_name":  "Tomato Target Spot",
        "pathogen_type": "Fungus",
        "pathogen":     "Corynespora cassiicola",
        "cause":        "Warm humid conditions; spreads in dense canopy",
        "region":       "South and Central India",
        "season":       "Monsoon",
        "symptoms":     "Concentric brown rings on leaves similar to early blight but smaller",
        "pesticide": [
            {"product": "Azoxystrobin 23% SC",     "dose": "1 ml/L water",   "when": "At first sign"},
            {"product": "Mancozeb 75% WP",          "dose": "2.5 g/L water",  "when": "Every 10 days"},
        ],
        "organic_option": "Prune lower leaves; improve plant spacing",
        "base_days": 10
    },
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": {
        "simple_name":  "Tomato Yellow Leaf Curl Virus",
        "pathogen_type": "Virus",
        "pathogen":     "Tomato yellow leaf curl virus (TYLCV)",
        "cause":        "Spread by whitefly Bemisia tabaci; no cure once infected",
        "region":       "All India",
        "season":       "Summer–Monsoon",
        "symptoms":     "Upward curling of young leaves; yellowing of leaf margins; stunted growth",
        "pesticide": [
            {"product": "Imidacloprid 17.8% SL (for whitefly)", "dose": "0.5 ml/L water", "when": "At first whitefly sign; every 14 days"},
            {"product": "Thiamethoxam 25% WG",                   "dose": "0.3 g/L water",  "when": "Alternate with imidacloprid"},
            {"product": "Yellow sticky traps",                   "dose": "10 traps/acre",  "when": "Continuous monitoring"},
        ],
        "organic_option": "Reflective mulch to repel whitefly; remove infected plants; plant TYLCV-resistant varieties",
        "base_days": 3
    },
    "Tomato___Tomato_mosaic_virus": {
        "simple_name":  "Tomato Mosaic Virus",
        "pathogen_type": "Virus",
        "pathogen":     "Tomato mosaic virus (ToMV)",
        "cause":        "Mechanical spread (tools, hands); infected seed",
        "region":       "All India",
        "season":       "All year",
        "symptoms":     "Mosaic yellow-green mottling on leaves; leaf distortion; stunted growth",
        "pesticide": [
            {"product": "No curative pesticide exists for viruses", "dose": "—", "when": "—"},
            {"product": "Disinfect tools with bleach 10% solution",  "dose": "Dip tools",  "when": "Before pruning each plant"},
        ],
        "organic_option": "Remove and burn infected plants; wash hands before touching plants; use certified virus-free seed",
        "base_days": 1
    },
    "Tomato___healthy": {
        "simple_name": "Healthy Tomato", "pathogen_type": "None",
        "pathogen": "None", "cause": "No disease", "region": "All",
        "season": "All year", "symptoms": "No symptoms",
        "pesticide": [{"product": "No pesticide needed", "dose": "—", "when": "—"}],
        "organic_option": "Calcium spray for blossom end rot prevention",
        "base_days": 30
    },

    # ── DEFAULT FALLBACK ──────────────────────────────────────
    "default": {
        "simple_name":  "Unknown Disease",
        "pathogen_type": "Unknown",
        "pathogen":     "Unknown",
        "cause":        "Could not identify specific pathogen",
        "region":       "India",
        "season":       "Monsoon",
        "symptoms":     "Visible leaf damage",
        "pesticide": [
            {"product": "Mancozeb 75% WP (broad spectrum)", "dose": "2.5 g/L water", "when": "As precaution"}
        ],
        "organic_option": "Consult your local Krishi Vigyan Kendra (KVK)",
        "base_days": 10
    }
}

print(f'✅ Disease info loaded for {len(DISEASE_INFO)-1} disease classes')

In [ ]:
# ============================================================
# CELL 15 — Weather-aware prediction function
# ============================================================
import numpy as np
import cv2
import matplotlib.pyplot as plt
import requests

# ---- Your Visual Crossing API key ----
VC_API_KEY = "VJ3KJEEDDQDGVS5MJPT2FAUDC"

def get_weather(city):
    try:
        url = (f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/"
               f"services/timeline/{city}?unitGroup=metric&key={VC_API_KEY}&contentType=json")
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return None
        today = r.json()["days"][0]
        return {
            "temp":        today["temp"],
            "humidity":    today["humidity"],
            "description": today["conditions"],
            "rain":        today["precip"] > 0
        }
    except Exception:
        return None


def get_spray_timing(disease_name, weather, base_days):
    if not weather:
        return f"Spray within {base_days} days. (Weather data unavailable)"
    h    = weather["humidity"]
    rain = weather["rain"]
    if "Late_blight" in disease_name or "mosaic_virus" in disease_name or "Yellow_Leaf_Curl" in disease_name:
        if rain or h > 85:
            return "⚠️ VERY URGENT: Act within 1–2 days! High humidity/rain increases spread rapidly."
        return f"Act within {base_days} days."
    if rain and h > 80:
        return f"Spray within {max(1, base_days-4)} days (High rain + humidity detected)."
    if h > 85:
        return f"Spray within {max(1, base_days-3)} days (High humidity detected)."
    return f"Spray within {base_days} days (Weather moderate)."


def predict_disease(img_path, city="Patna"):
    """Predict disease and show complete pesticide recommendations."""
    print('=' * 65)
    print(f'  LEAF DISEASE ANALYSIS')
    print(f'  Image : {img_path}')
    print(f'  City  : {city}')
    print('=' * 65)

    # Load image
    img = cv2.imread(img_path)
    if img is None:
        print('❌ Could not load image. Check path.')
        return

    # Preprocess — EfficientNet preprocessing
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_array = tf.keras.applications.efficientnet.preprocess_input(
        img_resized.astype('float32')
    )
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    preds = model.predict(img_array, verbose=0)[0]
    top3  = np.argsort(preds)[-3:][::-1]

    # Weather
    weather = get_weather(city)
    if weather:
        print(f'\n🌤️  Weather in {city}: {weather["temp"]}°C | '
              f'Humidity: {weather["humidity"]}% | {weather["description"]}'
              f'{" | 🌧️ Rain today" if weather["rain"] else ""}')
    else:
        print(f'\n⚠️  Weather data unavailable for {city}')

    print()

    # Display top 3 predictions
    for rank, idx in enumerate(top3, 1):
        disease_key  = CLASS_NAMES[idx]
        conf         = preds[idx] * 100
        info         = DISEASE_INFO.get(disease_key, DISEASE_INFO["default"])
        timing       = get_spray_timing(disease_key, weather, info["base_days"])
        is_healthy   = 'healthy' in disease_key.lower()
        status_emoji = '✅' if is_healthy else '🦠'

        print(f'{'='*65}')
        print(f'  #{rank}  {status_emoji}  {info["simple_name"]}  —  Confidence: {conf:.1f}%')
        print(f'{'='*65}')
        print(f'  Pathogen Type : {info["pathogen_type"]}')
        print(f'  Pathogen/Pest : {info["pathogen"]}')
        print(f'  Cause         : {info["cause"]}')
        print(f'  Symptoms      : {info["symptoms"]}')
        print(f'  Region        : {info["region"]}')
        print(f'  Season        : {info["season"]}')
        print(f'  ⏰ Timing     : {timing}')

        if not is_healthy:
            print(f'\n  🧪 PESTICIDE RECOMMENDATIONS:')
            for i, p in enumerate(info["pesticide"], 1):
                print(f'     {i}. {p["product"]}')
                print(f'        Dose : {p["dose"]}')
                print(f'        When : {p["when"]}')
            print(f'\n  🌱 ORGANIC OPTION:')
            print(f'     {info["organic_option"]}')
        else:
            print(f'  🌱 Tip: {info["organic_option"]}')

        print()

    # Bar chart
    diseases = [DISEASE_INFO.get(CLASS_NAMES[i], DISEASE_INFO['default'])['simple_name'].replace(' ', '\n') for i in top3]
    confs    = [preds[i] * 100 for i in top3]
    colors   = ['#4CAF50' if 'healthy' in CLASS_NAMES[i].lower() else '#E53935' for i in top3]

    plt.figure(figsize=(10, 5))
    bars = plt.bar(diseases, confs, color=colors, edgecolor='black', width=0.5)
    plt.ylabel('Confidence (%)', fontsize=12)
    plt.title('Top 3 Predictions', fontsize=14, fontweight='bold')
    plt.ylim(0, 105)
    plt.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, confs):
        plt.text(bar.get_x() + bar.get_width()/2, val + 1.5,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


print('✅ predict_disease() function ready!')
print()
print('Usage:')
print('  predict_disease(\'path/to/leaf.jpg\', city=\'Dhanbad\')')

In [ ]:
# ============================================================
# CELL 16 — Run prediction on your image
# ============================================================
# Change the path below to your actual leaf image.
# For Colab, you can upload an image with:
#   from google.colab import files
#   uploaded = files.upload()
# Then use the filename as the path.

from google.colab import files
print('Upload a leaf image to test:')
up = files.upload()

for filename in up.keys():
    predict_disease(filename, city='Dhanbad')   # change city to yours

---
## After Training: Update app.py on HuggingFace

### Change 1 — Fix preprocessing in `preprocess_for_disease()`:

```python
def preprocess_for_disease(image: Image.Image) -> np.ndarray:
    img = image.convert('RGB')
    # DELETE the border crop lines — they cut off leaf scorch symptoms
    img = img.resize((224, 224))
    # Use EfficientNet preprocessing (NOT /255.0)
    arr = tf.keras.applications.efficientnet.preprocess_input(
        np.array(img, dtype='float32')
    )
    return arr
```

### Change 2 — Update CLASS_NAMES in app.py:
Use the list from `class_names.json` that you downloaded.

### Change 3 — Upload to HuggingFace:
1. Go to your Space → Files → Upload files
2. Upload `plant_disease_model_final.h5` (replaces old one)
3. Upload `class_names.json`
4. Edit `app.py` with the preprocessing fix above
5. Space will auto-restart and use the new model

---
## Summary: Do you need 10 epochs?

| What | Answer |
|------|--------|
| Should you use exactly 10 epochs? | **No** — 10 epochs causes underfitting |
| How many epochs will this notebook run? | **Automatically decided by EarlyStopping** |
| Typical total epochs (Phase 1 + Phase 2)? | **12–20 total** |
| Does it waste time training extra? | **No** — stops when accuracy stops improving |
| Expected final accuracy? | **93–96%** (vs ~78% with old 10-epoch MobileNetV2) |